# EUPPBench station network: the paper figures

Produces the EUPPBench half of Figure 3 of the paper and the double-column ETTh2 + EUPPBench
overview it is drawn as (§7), plus appendix-style EUPPBench map figures. "Fig. N" below
refers to this notebook's own figure numbering, not the paper's.

Post-processing of ensemble 2-m temperature forecasts at **115 European stations** with the
multigraph GAT backbone of Feik, Lerch & Stühmer (2024,
[arXiv:2407.11050](https://arxiv.org/abs/2407.11050)), on the reforecast-to-forecast (R2F) task at
**72 h lead**: fit on the 11-member reforecasts (1997–2009), evaluated on the 51-member 2017–2018
forecasts (730 daily cases). One forecast case is a single 115-dimensional Gaussian over the
station network, so the **dense Σ is available** — nothing like the 2016² of the time-series tasks
or the 35 200² of ERA5 — and every panel below works with Σ itself.

Five arms, same backbone, same 115-dimensional target, differing only in the predictive
covariance. Every arm but the Dirac is fitted on the **kernel score**, so a difference between two
panels is a difference of *parameterization*, not of objective:

| arm | Σ | fitted with |
|---|---|---|
| Det | — (Dirac at μ) | MSE, single stage |
| LorD (r = 1) | `diag(d) + u uᵀ` | kernel score, single stage |
| Chol | `L Lᵀ`, unstructured — the ceiling any parameterization is measured against, tractable here because V = 115 | kernel score, single stage |
| SB | none: the backbone maps `(x, latent) → one member`, and Σ is the **sample** covariance of M = 100 members | fair ensemble kernel score, single stage |
| SCORE (graph Fourier transform, Matérn filter) | `D̃ (Bᵀ diag(s) B) D̃`, `B` the graph Fourier basis, `s` a Matérn spectrum on the Laplacian eigenvalues | two-stage fit: stage 1 marginal CRPS, stage 2 dense kernel score on the predictive covariance `Σ_P` |

Every figure is written at the paper's column width by `notebooks/plot_config.py` and saved as a
PDF that goes into LaTeX with `\includegraphics[width=\linewidth]{...}` — no rescaling, no titles.
Panels are identified by axis labels, row labels and colorbar labels only; whatever else a reader
needs belongs in the figure caption.

**One real complication: missing observations.** 0.88 % of test station-days are unreported and
only 25.5 % of the 730 test days are complete, so no empirical estimate may use a plain `mean`:
empirical covariances are pairwise-complete and the joint calibration carries a per-day degrees of
freedom (§3).

## 0 Setup

In [ ]:
import json
import math
import os
import sys
from pathlib import Path

# --- Project-root bootstrap: put the repo root on sys.path so `import src...`
# works, and export PROJECT_ROOT for any Hydra ${oc.env:PROJECT_ROOT} lookups. ---
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".project-root").exists()),
    Path.cwd(),
)
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("PROJECT_ROOT", str(PROJECT_ROOT))

import hydra
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.collections import LineCollection
from omegaconf import OmegaConf
from src.utils.legacy import upgrade_legacy_targets
from scipy import stats as sps

from src.methods import PredictiveMethod
from src.methods.base import _as_list
from src.methods.transforms import graph_laplacian_eigenbasis
from src.models.gaussian_predictive import LightningPredictiveModel

# Cartopy draws the land/sea/border context under the stations (as in
# compare_era5_cov.ipynb). It is not in requirements.txt, so the maps degrade to
# plain lat/lon axes rather than failing if it is missing.
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except ImportError:  # pragma: no cover - environment-dependent
    HAS_CARTOPY = False

# --- shared figure configuration: paper widths, fonts, colors (notebooks/plot_config.py) ---
if str(PROJECT_ROOT / "notebooks") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))
import plot_config as pc

PLOT = pc.apply_style()      # scienceplots + the paper's font sizes / line widths / rcParams
FIG_DIR = pc.FIG_DIR

# Colormaps by role, all from plot_config so a panel here matches the same quantity elsewhere in
# the paper: absolute temperature, signed departures (anomaly / residual / correlation), and
# non-negative magnitudes (sigma).
CMAP_T = pc.temperature_cmap()
CMAP_SIGNED = pc.diverging_cmap()
CMAP_MAG = pc.sequential_cmap()
CMAP_EDGE = pc.grey_ramp()   # correlation edges: light = weak, dark = strong

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(sys.executable)
print("project root:", PROJECT_ROOT, "| device:", DEVICE,
      "| cartopy:", "yes" if HAS_CARTOPY else "no (plain lat/lon axes)")
print("figures ->", FIG_DIR)

## 1 Configuration

`RUNS` maps a display name to a **run folder**, not to a checkpoint file: the folder carries the
resolved config the rebuild in §2 needs, and `find_checkpoint` picks the deployed weights out of
it. Every arm lives in the results tree under
`station_postprocessing/euppbench/<slug>/seed<N>/`,
with `config.yaml` next to `model.ckpt` (single-stage) or `stage1.ckpt`/`stage2.ckpt` (two-stage
fit). The raw Hydra
layout (`.hydra/config.yaml` + `checkpoints/`) is still supported by the loader, so a
not-yet-synced local run can be dropped into `RUNS` unchanged.

All five slugs carry **seeds 1–5**, so `SEED` switches every arm together.

Two slugs need reading:

* `whitened_graph_matern_s1crps_dense_kernel_score` is SCORE. `_s1crps` is a stage-1 marginal
  CRPS, and `_dense_kernel_score` means stage 2 scores the predictive covariance `Σ_P` against
  `y − μ` directly: on a *localized* basis the renormalization by the implied marginal variances
  Δ(Σ̂) makes `D̃` depend on the stage-2 parameter, so scoring the core against a fixed
  standardized target would not be a proper score for `Σ_P`.
* `generative_kernel_score` is SB, fitted on the *fair* ensemble kernel score — the same
  estimator the reported metric uses, which is what puts it on this all-KS row.

`EDGE_ARMS` and `DRAW_ARMS` are the arm sets of the two multi-arm figures, and `EDGE_CASE` /
`DRAW_CASE` / `GRID_CASE` / `MATRIX_CASE` are their forecast cases — every single-case figure
takes a plain integer, so a case is changed by editing one number here (or by passing it at the
call site) rather than by re-running the loader.

In [ ]:
from src.utils.paths import configured_path
SHARED_CKPT = configured_path("results_dir") / "station_postprocessing/euppbench"
SEED = 1                                               # every slug carries seeds 1-5
WHITENED_SLUG = "whitened_graph_matern_s1crps_dense_kernel_score"

DETERMINISTIC = "Deterministic"
LORA = "LoRa (r=1)"
CHOLESKY = "Full cov."
GENERATIVE = "Generative"
OURS = "Ours"

RUNS = {
    # experiments=train/euppbench  method=deterministic  method.loss=mse -- the point
    # forecast: no covariance at all, so it appears only in the panels a Dirac can fill.
    DETERMINISTIC: SHARED_CKPT / f"deterministic_mse/seed{SEED}",
    # method=phys_lora  method.loss=kernel_score  (rank=1 is the default): LorD, diag + a
    # single global mode, fitted on the same objective as every other arm but the Dirac.
    LORA: SHARED_CKPT / f"phys_lora_r1_kernel_score/seed{SEED}",
    # method=phys_cholesky  method.loss=kernel_score: Chol, an unstructured Sigma = L Lᵀ, the
    # ceiling any structured parameterization is measured against. Affordable only because
    # V = 115 (Cholesky.max_event_dim = 1024).
    CHOLESKY: SHARED_CKPT / f"phys_cholesky_kernel_score/seed{SEED}",
    # method=generative  method.loss=kernel_score: SB, no distributional assumption at all --
    # the backbone maps (x, latent) -> one member and Sigma is the sample covariance of the
    # M = 100 members it returns. See §5 for what that costs.
    GENERATIVE: SHARED_CKPT / f"generative_kernel_score/seed{SEED}",
    # experiments=train/euppbench_score: SCORE -- standardized targets, graph Fourier basis,
    # three-scalar Matern filter on the Laplacian eigenvalues, two-stage fit.
    OURS: SHARED_CKPT / f"{WHITENED_SLUG}/seed{SEED}",
}

# Paper wording, matching the arm names in calibration_histograms.ipynb and the results tables.
LABELS = {DETERMINISTIC: "Det", LORA: r"LorD $r=1$", CHOLESKY: "Chol",
          GENERATIVE: "SB", OURS: "SCORE"}
# Space-saving variant for panel tags -- no "$\mu$," prefix and no rank suffix, so
# "(b) $\mu$, LorD $r=1$" becomes "(b) LorD" (Fig. 1's single-column version needs the room).
SHORT_LABELS = {DETERMINISTIC: "Det", LORA: "LorD", CHOLESKY: "Chol",
                GENERATIVE: "SB", OURS: "SCORE"}
# One color per *arm*, not per objective: four of the five arms are fitted on the kernel score,
# so plot_config's objective roles cannot separate them. Det and SCORE keep the roles
# they have elsewhere in the paper; the physical-space family shares the low-rank ramp
# (LorD light, Chol dark) and SB takes the remaining accent.
ARM_COLORS = {
    DETERMINISTIC: pc.COLORS["deterministic"],
    LORA: pc.COLORS["lora"],
    CHOLESKY: pc.COLORS["lora_dark"],
    GENERATIVE: pc.COLORS["energy_score"],
    OURS: pc.COLORS["kernel_score"],
}

# Which arms each multi-arm figure shows. Fig. 1 draws correlation *edges*, so it needs a
# closed-form Sigma whose off-diagonal is a parameter, not a 100-member sample estimate. The
# main-text version leaves out Chol (and SB, which has no fitted off-diagonal at all) so the
# figure is three panels at the single-column width; the appendix version
# (`plot_truth_and_edges_grid`) adds both back in -- Chol as a fitted unstructured Sigma, SB
# as the *sample* correlation of its M members, which `full_covariance` already estimates
# from the same forward pass. Fig. 4 draws *members*, which every arm with a covariance can
# produce.
EDGE_ARMS = [LORA, OURS]
EDGE_ARMS_APPENDIX = [LORA, CHOLESKY, GENERATIVE, OURS]
DRAW_ARMS = [LORA, CHOLESKY, GENERATIVE, OURS]

# The forecast case of each single-case figure, as a plain index into the test split. The split
# is 730 daily forecasts covering 2017-01-01 ... 2018-12-31 in order and the loader does not
# shuffle, so an index is a date: 0 -> 2017-01-01, 182 -> 2017-07-02, 365 -> 2018-01-01,
# 547 -> 2018-07-02. Every figure below also takes `case_index` at the call site, so a second
# day can be rendered without touching these.
CASE_INDEX = 0          # the default, and what §5's summary and the network panel report
EDGE_CASE = CASE_INDEX      # Fig. 1, the correlation-edge panels
GRID_CASE = CASE_INDEX      # Fig. 2, mu / sigma / z per arm
DRAW_CASE = CASE_INDEX      # Fig. 4, draws against the realized error
MATRIX_CASE = CASE_INDEX    # Fig. 5, the correlation matrices
# Four days spread over both test years, for the multi-case grid.
CASE_INDICES = [0, 182, 365, 547]
# The first three of those days -- both winters and the summer between them -- as the rows of
# the appendix version of Fig. 1.
EDGE_CASES_APPENDIX = CASE_INDICES[:3]
N_DRAWS = 4             # ensemble members drawn per case for the realization panels

# Which station pairs count as "high correlation" in the edge panels. An absolute cut cannot be
# fixed in advance -- it has to sit inside the range of correlations the trained model actually
# produces, which changes with the arm, the case and a retrain. `RHO_QUANTILE` of *that arm's*
# off-diagonal |rho| is that cut, so every panel draws the same number of pairs; set
# `RHO_THRESHOLD` to a number to override it with one absolute cut everywhere.
RHO_QUANTILE = 0.95
RHO_THRESHOLD = None


# PlateCarree measures both axes in degrees, so at 49.6 degN the domain is drawn
# 1/cos(lat) = 1.55x wider than it is: 1.0 leaves a panel in raw degrees, 1.55 is the
# geographically faithful shape. The main figure stretches its panels part of the way
# back; the multi-panel grids stay at 1.0, where the same correction would cost them a
# full page of height.
MAIN_FIG_ASPECT = 1.25

GRAPH_MAX_DIST = 100.0  # the backbone's message-passing cutoff, in km
MIN_PAIR_DAYS = 50      # station pairs with fewer common observed days are not estimated
MAX_TEST_BATCHES = None  # None = the whole 730-day test split; an int truncates
# The test split carries 51 ensemble members (the reforecasts have 11), so a forward pass builds
# ~355k graph edges per case. 8 -- the datamodule's own eval_batch_size -- fits in 24 GB; 32 does not.
EVAL_BATCH_SIZE = 8

# Map background. "10m" is the Natural Earth resolution -- the right one for a 9-degree-wide
# domain. Once cached in cartopy's data directory nothing is downloaded. PlateCarree matches
# compare_era5_cov.ipynb and keeps the tick labels horizontal across a grid of panels.
MAP_SCALE = "10m"
PROJECTION = ccrs.PlateCarree() if HAS_CARTOPY else None

SAVE_FIGS = True        # pc.save writes <FIG_DIR>/<name>.pdf at exactly the paper size


def savefig(fig, name):
    """Write `fig` into notebooks/figures/ at the paper size, when SAVE_FIGS is on."""
    if not SAVE_FIGS:
        return None
    return pc.save(fig, f"{name}.pdf")


for _name, _rd in RUNS.items():
    print(f"{LABELS[_name]:<16s} {_rd}")

## 2 Rebuilding a run from its checkpoint

The checkpoints are saved with `save_weights_only=True` and neither the backbone nor the method is
in them (both are excluded from `save_hyperparameters`), so both are re-instantiated from the
logged config and the state dict is loaded into a plain `LightningPredictiveModel`. Staging only
affects *training*; the state dict is `backbone.*` either way.

Four things are runtime-derived and would silently differ if rebuilt by hand — the last two are
what an arm-set this wide adds, and each of them is a hard failure rather than a quiet one, which
is the good case:

* **`gamma`.** The kernel-score bandwidths are `sqrt(factor · d)` with `d = dm.get_total_dimension()`,
  which is **115** here — so `gamma_train = [5.36, 7.58, 10.72, 15.17, 23.98]` and
  `gamma_eval = [10.72]`. Only the scores use them, but a wrong `gamma` changes the score table.
* **`group_widths` / `group_denorm` / `other_zero_init`.** The head geometry is a property of the
  *method* (Det 1 output, LorD 3, Chol `V + 1 = 116`, SB one per
  member, SCORE 3 with a global spectrum readout), so it is read off the instantiated
  method rather than written in the backbone config.
* **`event_dim`.** `Cholesky` sizes its head as one row of `L` per coordinate, so it has to
  know `V` *before* the backbone is built and refuses to instantiate without it. It is the same
  `dm.get_total_dimension()` the bandwidths come from, and `evaluate.py` / `calibration.py` inject
  it the same way; every other method absorbs it and ignores it.
* **`noise`.** SB's randomness lives *inside* the network — the backbone splices a
  `GaussianNoiseInjector` between its trunk and its head — so `noise=method.noise` has to reach
  the backbone or the injector's weights have nowhere to load into (`Unexpected key(s) in
  state_dict: backbone.noise_injector.*`). It is `None` for every closed-form family, which draws
  its members *after* the head.

`load_run_config` also makes `datamodule.data_path` absolute: the training job logged it relative
to the repo root and a notebook's cwd is `notebooks/`. That has to happen inside the config rather
than at the call site, since the station distances reach the backbone, SCORE's graph transform and
the variogram score through `${datamodule.data_path}` interpolations.

In [ ]:
_DM_CACHE, _MODEL_CACHE = {}, {}


def load_run_config(run_dir):
    """The resolved config of a run, from either folder layout, with `data_path` absolute.

    `config.yaml` at the top of a synced seed folder, `.hydra/config.yaml` in a raw
    Hydra run directory -- the two are the same object, written by the same job.
    """
    run_dir = Path(run_dir)
    for candidate in (run_dir / "config.yaml", run_dir / ".hydra" / "config.yaml"):
        if candidate.exists():
            cfg = upgrade_legacy_targets(OmegaConf.load(candidate))
            break
    else:
        raise FileNotFoundError(f"No config.yaml or .hydra/config.yaml under {run_dir}")
    data_path = cfg.datamodule.get("data_path")
    if data_path is not None and not os.path.isabs(data_path):
        cfg.datamodule.data_path = str(PROJECT_ROOT / data_path)
    return cfg


def find_checkpoint(run_dir):
    """The *deployed* weights of a run, in both layouts.

    Synced seed folder: `stage2.ckpt` (the deployed stage of a two-stage run) before
    `model.ckpt` (single stage). Hydra run directory: `checkpoints/stage2/*.ckpt`
    before `checkpoints/*.ckpt`. Never `stage1.ckpt`, which is a training artefact.
    """
    run_dir = Path(run_dir)
    for name in ("stage2.ckpt", "model.ckpt"):
        if (run_dir / name).exists():
            return str(run_dir / name)
    ck = run_dir / "checkpoints"
    found = sorted(ck.glob("stage2/*.ckpt")) or sorted(ck.glob("*.ckpt"))
    if not found:
        raise FileNotFoundError(f"No checkpoint under {run_dir}")
    return str(found[-1])


def load_metrics(run_dir):
    """The synced `metrics.json` test scores, or `None` for a run that has no such file."""
    path = Path(run_dir) / "metrics.json"
    return json.loads(path.read_text())["test"] if path.exists() else None


def build_datamodule(cfg, num_workers=0, batch_size=EVAL_BATCH_SIZE, use_cache=True):
    """The run's datamodule with the test split set up (no worker processes).

    Cached on the data-shaping fields, so the arms -- which share one dataset --
    memory-map the arrays once.
    """
    dm_cfg = cfg.datamodule
    key = (dm_cfg._target_, dm_cfg.get("data_path"), batch_size)
    if use_cache and key in _DM_CACHE:
        return _DM_CACHE[key]
    dm = hydra.utils.instantiate(dm_cfg, num_workers=num_workers,
                                 eval_batch_size=batch_size,
                                 persistent_workers=False)
    dm.setup("test")
    if use_cache:
        _DM_CACHE[key] = dm
    return dm


def build_model_from_run(run_dir, device=DEVICE, use_cache=True):
    """Rebuild `(model, method, datamodule, cfg, ckpt_path)` for a run folder."""
    if use_cache and str(run_dir) in _MODEL_CACHE:
        return _MODEL_CACHE[str(run_dir)]
    cfg = load_run_config(run_dir)
    dm = build_datamodule(cfg)

    d = dm.get_total_dimension()
    gamma_train = [math.sqrt(f * d) for f in _as_list(cfg.get("gamma_train", [1.0]))]
    gamma_eval = [math.sqrt(f * d) for f in _as_list(cfg.get("gamma_eval", [1.0]))]

    # `event_dim` is what Cholesky sizes its head with, and it refuses to be built
    # without it; every other method takes it through **kwargs and ignores it.
    method: PredictiveMethod = hydra.utils.instantiate(
        cfg.method, out_channels=dm.get_channels(), event_dim=d,
        gamma=gamma_train, gamma_eval=gamma_eval)
    # `noise` is None for every closed-form family and a spec for SB, whose backbone
    # carries a noise injector between trunk and head. Omitting it loads no weights
    # into that injector and fails loudly on `backbone.noise_injector.*`.
    backbone = hydra.utils.instantiate(
        cfg.backbone, in_channels=dm.get_input_channels(),
        out_channels=method.out_channels, group_widths=method.group_widths,
        group_denorm=method.group_denorm, other_zero_init=method.other_zero_init,
        noise=method.noise)

    ckpt = find_checkpoint(run_dir)
    state = torch.load(ckpt, map_location="cpu", weights_only=False)["state_dict"]
    model = LightningPredictiveModel(backbone=backbone, predictive_method=method)
    model.load_state_dict(state)
    model.eval().to(device)
    method.training = False  # eval mode -> gamma_eval / n_eval_samples

    out = (model, method, dm, cfg, ckpt)
    if use_cache:
        _MODEL_CACHE[str(run_dir)] = out
    return out

## 3 What we extract

**The covariance.** No method exposes Σ — deliberately, since a dense Σ is 4.6 GiB at ERA5's
`d = 35 200`. What every family does expose is its marginal variance and its *pairwise difference*
variance (the two quantities the variogram score needs), and those determine Σ:

$$\Sigma_{ij} \;=\; \tfrac12\big(\Sigma_{ii} + \Sigma_{jj} - \operatorname{Var}(X_i - X_j)\big).$$

`pair_diff_var(pred, rows=arange(115))` returns the whole `(B, 115, 115)` block in one call, so
`full_covariance` is exact for all five arms and costs nothing. It is the same identity
`PredictiveMethod._dense_score_terms` uses to build the matrix the masked scores are taken
against, so the Σ plotted here is the Σ the scores see.

Note what this does *not* do: it never asks a family how it stores its covariance. The
Det arm returns zeros, LorD a diagonal plus a rank-1 outer product, Chol
`L Lᵀ`, SB the **sample** covariance `A Aᵀ/(M−1)` of its 100 members, and
SCORE `D̃ Bᵀ diag(s) B D̃` — all through the same two accessors. The deterministic arm is
therefore a free end-to-end test of the extraction path: a Dirac must come back with Σ ≡ 0, and it
is the reason every figure below carries a `has_cov` branch rather than a special case for one arm.

SB is the arm whose Σ is an *estimate* rather than a parameter, and it is
structurally singular: 100 members in 115 dimensions give a rank-99 covariance, so its smallest
eigenvalue is 0 up to rounding and it has no Cholesky. That is a property of the model, not of the
extraction — it is why it appears in the draw panels (§6.4) and the correlation matrices (§6.5),
where only second moments are needed, and not in the edge panels (§6.1) or anywhere a `Σ⁻¹` is
taken.

**Missing observations.** `y` is `(B, 1, 115)` in °C with NaNs preserved. Two masked estimators do
all the work:

* `pairwise_complete_stats` — second moments over the days where *both* stations reported. They
  are **uncentered** on purpose: the predictive claims `r = y − μ ~ N(0, Σ)`, so a bias in μ is a
  real deficiency and should show up here rather than be subtracted away. The result can be
  indefinite, so it is only ever plotted, never inverted.
* `masked_maha` — the Mahalanobis norm of one case restricted to that day's observed stations,
  which for a Gaussian is exactly the predictive marginal on the observed set (the same reduction
  `masked_gaussian_nll` performs). Its degrees of freedom is the day's observed count, so it
  varies from case to case. It returns `NaN` when the restricted Σ has no Cholesky, which is
  every day for SB and no day for the four closed-form ones — §5 prints that count.

In [ ]:
def full_covariance(method, pred):
    """Dense (B, V, V) predictive covariance from the family's closed forms.

    Sigma_ij = 0.5 (Sigma_ii + Sigma_jj - Var(X_i - X_j)); the diagonal comes from
    `get_marg_std` and is exact on every path.
    """
    var = method.get_marg_std(pred).flatten(start_dim=1).pow(2)      # (B, V)
    rows = torch.arange(var.shape[1], device=var.device)
    pair = method.pair_diff_var(pred, rows)                          # (B, V, V)
    cov = 0.5 * (var.unsqueeze(-1) + var.unsqueeze(-2) - pair)
    return 0.5 * (cov + cov.transpose(-1, -2))  # symmetrize away rounding


def cov_to_corr(cov, eps=1e-12):
    """Correlation matrix; NaN wherever a marginal variance is zero (a Dirac's Sigma)."""
    d = np.sqrt(np.clip(np.diag(cov), 0.0, None))
    denom = np.outer(d, d)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(denom > eps, cov / np.where(denom > eps, denom, 1.0), np.nan)


def off_diagonal(corr):
    """The upper-triangle correlations of one case, NaNs dropped."""
    iu = np.triu_indices(corr.shape[0], k=1)
    rho = np.asarray(corr, dtype=np.float64)[iu]
    return rho[np.isfinite(rho)]


def pairwise_complete_stats(resid, min_count=MIN_PAIR_DAYS):
    """Uncentered pairwise-complete second moments of a (N_days, V) residual matrix.

    Returns `(S, n)` with `S_ij = E[r_i r_j]` and `n_ij` the number of days on which
    both stations reported. Both fall out of two matmuls on the NaN-zeroed residual
    and its finite mask, so no pair loop is needed. Pairs seen on fewer than
    `min_count` common days are returned as NaN rather than as a noise estimate off a
    handful of days.
    """
    finite = np.isfinite(resid)
    F = finite.astype(np.float64)
    R = np.where(finite, resid, 0.0).astype(np.float64)
    n = F.T @ F                     # common-day counts
    cross = R.T @ R                 # sum r_i r_j over common days
    enough = n >= max(min_count, 1)
    with np.errstate(invalid="ignore", divide="ignore"):
        S = np.where(enough, cross / np.maximum(n, 1.0), np.nan)
    return S, n


def masked_maha(cov, resid, jitter_rel=1e-8):
    """`(maha, dof)` for one case, restricted to the stations that reported.

    A Gaussian's marginal on a coordinate subset is the corresponding sub-block, so
    restricting Sigma and r to the observed rows is exact -- not an approximation.
    The jitter is the ts notebooks' `1e-8 * mean(diag)`.
    """
    obs = np.isfinite(resid)
    d = int(obs.sum())
    if d == 0:
        return np.nan, 0
    c = cov[np.ix_(obs, obs)].astype(np.float64)
    r = resid[obs].astype(np.float64)
    c = c + np.eye(d) * (jitter_rel * float(np.diag(c).mean()))
    try:
        L = np.linalg.cholesky(c)
    except np.linalg.LinAlgError:
        return np.nan, d
    z = np.linalg.solve(L, r)
    return float(z @ z), d

## 4 Station geometry and map primitives

Geometry comes from the dataset's `meta.npz`, via `dm.get_graph()` → `(dist, coords)` with `dist`
the **geodesic** (WGS-84, not haversine) station separation in km and `coords` the `(115, 2)`
lat/lon array. That is the same file the backbone reads for its edges and the variogram score for
its inverse-distance weights, so the geometry has one source. It never goes through the
projection: the projection only decides where a dot is drawn.

The panel helpers are what makes these figures paper figures rather than notebook figures:

* **Sizes come from `plot_config`.** `map_grid` derives the figure height from the *domain* aspect
  (the extent is 9.0° × 8.5°, so a panel is nearly square) times the number of rows, at the
  single- or double-column width. Fixed-aspect GeoAxes otherwise leave a band of white space that
  `bbox_inches="tight"` would have to crop — which would change the width the paper asked for.
* **No titles.** `map_axes` takes no title argument at all. Rows are named by `row_label` (rotated
  text left of the leftmost panel), columns by their colorbar label, and panels by an optional
  `panel_tag` — `(a)`, `(b)`, … — that a caption can refer to.
* **Resolution.** `MAP_SCALE = "10m"` is right for a 9°-wide domain and is what is cached locally.
  The module-level `cfeature.LAND` / `COASTLINE` / `BORDERS` are 110m — both too coarse here and
  *not* in the cache, so they would try to hit the network. Everything goes through
  `.with_scale(MAP_SCALE)`, and the feature objects are built once and shared across all panels so
  the shapefiles are parsed a single time.
* **Coordinates.** Station positions are always plain lat/lon, so every artist built from them
  carries `transform=ccrs.PlateCarree()` (bundled as `TRANSFORM`) regardless of the axes
  projection. Getting this wrong silently plots degrees as projection metres.

Two structural facts the maps mark explicitly:

* **Three stations are isolated** under the backbone's 100 km message-passing cutoff (degree
  min / median / max = 0 / 9 / 25 over 1216 directed edges). They receive no spatial messages, so
  they are the natural place for the GNN to fail. They are ringed in black **only in the network
  panel** (§6.7), where the graph that makes them isolated is drawn with them: on a panel without
  that graph a ring has nothing to refer to and reads as a flagged observation instead.
* **The station axis is sorted by country** (AT/BE/FR/DE/NL), not by geography, so a covariance
  heatmap in the native order is unreadable. `fiedler_order` re-sorts by the Fiedler vector of the
  *same* dense Gaussian-kernel Laplacian SCORE's graph Fourier basis is built from — so the matrix
  is ordered by the graph's own leading geometry, not by an ad-hoc lat/lon sort.

In [ ]:
def fiedler_order(dist, length_scale=None, knn=10):
    """Station permutation sorted along the Fiedler vector of the GFT Laplacian.

    Reuses `graph_laplacian_eigenbasis` (rows = eigenvectors, ascending eigenvalue) so
    the ordering is the one the whitened method's basis induces: row 0 is the
    near-constant mode, row 1 the Fiedler vector, whose ordering is the classical
    spectral 1-D embedding of the network.
    """
    basis = graph_laplacian_eigenbasis(
        torch.as_tensor(np.asarray(dist), dtype=torch.float64),
        length_scale=length_scale, knn=knn, normalization="sym",
    )
    return np.argsort(basis[1].cpu().numpy())


def station_geometry(dm, max_dist=GRAPH_MAX_DIST):
    """Everything the map panels need, from the dataset metadata."""
    meta = dm.meta
    dist, coords = dm.get_graph()
    dist = np.asarray(dist, dtype=np.float64)
    src, _ = np.nonzero((dist > 0.0) & (dist <= max_dist))
    degree = np.bincount(src, minlength=dist.shape[0])
    lat, lon = coords[:, 0], coords[:, 1]
    return {
        "dist": dist, "lat": lat, "lon": lon,
        "alt": np.asarray(meta["alt"]),
        "name": np.asarray(meta["station_name"]),
        "sid": np.asarray(meta["station_id"]),
        "times": pd.to_datetime(meta["test_time"]),
        "degree": degree, "isolated": degree == 0,
        "n_edges": int(src.size), "graph_max_dist": max_dist,
        "order": fiedler_order(dist),
    }


# Feature objects are built once and reused on every panel: cartopy caches the parsed
# geometries on the object, so sharing them turns ~40 shapefile reads into one.
if HAS_CARTOPY:
    _OCEAN = cfeature.OCEAN.with_scale(MAP_SCALE)
    _LAND = cfeature.LAND.with_scale(MAP_SCALE)
    _COAST = cfeature.COASTLINE.with_scale(MAP_SCALE)
    _BORDERS = cfeature.BORDERS.with_scale(MAP_SCALE)

# Station coordinates are always plain lat/lon, whatever the axes projection is, so
# every artist drawn from them carries a PlateCarree transform.
TRANSFORM = {"transform": ccrs.PlateCarree()} if HAS_CARTOPY else {}


def map_extent(geo, pad_lon=0.6, pad_lat=0.4):
    """`[lon0, lon1, lat0, lat1]` around the station network, with a small margin."""
    return [geo["lon"].min() - pad_lon, geo["lon"].max() + pad_lon,
            geo["lat"].min() - pad_lat, geo["lat"].max() + pad_lat]


def map_grid(geo, nrows, ncols, columns=None, extra_h=0.0, aspect_scale=1.0, **kwargs):
    """A grid of map panels at the paper width, sized so the panels tile without gaps.

    The panel height/width ratio is the domain's, because both PlateCarree GeoAxes and
    the no-cartopy fallback keep a fixed aspect; `extra_h` (inches) is room for
    colorbars and tick labels, which live outside the panels. `aspect_scale` is the
    latitude stretch the panels are drawn with (see `stretch_domain`) -- passing the same
    number here is what keeps the figure height and the panel shape consistent.
    """
    width = pc.width_of(columns)
    extent = map_extent(geo)
    ratio = aspect_scale * (extent[3] - extent[2]) / (extent[1] - extent[0])
    height = nrows * ratio * (width / ncols) + extra_h
    kwargs.setdefault("subplot_kw", {"projection": PROJECTION} if HAS_CARTOPY else {})
    return pc.subplots(nrows, ncols, aspect=height / width, columns=columns,
                       squeeze=False, **kwargs)


def map_axes(ax, geo, xlabels=False, ylabels=False):
    """Domain furniture for a station panel: extent, land/sea, borders, graticule.

    Labels are off by default -- on a paper figure the coastline localizes the domain
    and the graticule labels are clutter -- and switchable so a grid can put them on
    the bottom row and left column only.
    """
    extent = map_extent(geo)
    if not HAS_CARTOPY:
        # No cartopy: plain lat/lon axes, with the aspect corrected for latitude so the
        # domain is not stretched by the usual 1/cos(lat) factor.
        ax.set_aspect(1.0 / np.cos(np.deg2rad(float(geo["lat"].mean()))))
        ax.set_xlim(extent[0], extent[1])
        ax.set_ylim(extent[2], extent[3])
        ax.set_xlabel("longitude [$^\\circ$E]" if xlabels else "")
        ax.set_ylabel("latitude [$^\\circ$N]" if ylabels else "")
        ax.grid(alpha=0.15)
        return ax

    ax.set_extent(extent, crs=ccrs.PlateCarree())
    # Muted fills: the station colors carry the signal, the geography is context.
    # `rasterized=True` is not cosmetic: the 10m Natural Earth polygons are ~8 MB of
    # vector geometry *per panel*, so a 16-panel figure would be a >100 MB PDF.
    # Rasterized at savefig.dpi they cost a few hundred kB and, being pure
    # background, lose nothing; the stations, edges and text stay vector.
    ax.add_feature(_OCEAN, facecolor=pc.MAP_COLORS["ocean"], zorder=0, rasterized=True)
    ax.add_feature(_LAND, facecolor=pc.MAP_COLORS["land"], zorder=0, rasterized=True)
    ax.add_feature(_COAST, edgecolor=pc.MAP_COLORS["coast"], rasterized=True,
                   linewidth=PLOT.thin_line_width * 0.6, zorder=1)
    ax.add_feature(_BORDERS, edgecolor=pc.MAP_COLORS["border"], rasterized=True,
                   linewidth=PLOT.thin_line_width * 0.6, zorder=1)

    draw = {}
    if xlabels:
        draw["bottom"] = "x"
    if ylabels:
        draw["left"] = "y"
    gl = ax.gridlines(draw_labels=draw or False, x_inline=False, y_inline=False,
                      linewidth=0.3, color=pc.MAP_COLORS["graticule"], alpha=0.4)
    gl.xlabel_style = gl.ylabel_style = {"size": PLOT.annotation_size}
    return ax


def stretch_domain(ax, scale):
    """Draw a PlateCarree panel taller than raw degrees, towards the domain's true shape.

    Cartopy fixes a GeoAxes' aspect to 1 in *projection* units, and PlateCarree's units
    are degrees, so a mid-latitude domain comes out too wide by 1/cos(lat). Overriding
    the aspect is the cheapest fix that keeps the projection, the lat/lon graticule and
    the horizontal tick labels; a conformal projection would be the other way to do it.
    The no-cartopy fallback already applies the full 1/cos(lat) correction itself.
    """
    if HAS_CARTOPY and scale != 1.0:
        ax.set_aspect(scale)


def sync_colorbar_height(fig, cbar, ref_ax):
    """Snap a `location="right"` colorbar's vertical extent to `ref_ax`'s real box.

    Constrained layout sizes a "right" colorbar to its row's nominal grid-cell height, not
    to what is left once a fixed-aspect axes (a `stretch_domain`-ed GeoAxes, here) has
    shrunk within that cell to keep its aspect -- so an unfixed bar over/undershoots the
    panels it labels by the same margin on both ends. One draw lets the aspect correction
    and the constrained-layout solve settle (including the room any other colorbar already
    claimed), so `ref_ax.get_position()` right after it is the panels' true box; the layout
    engine is then switched off so this fix is not undone by a later re-solve (`savefig`
    draws again). Only the height moves -- `cbar.ax`'s x-position and width, which
    constrained layout placed correctly, are kept.
    """
    fig.canvas.draw()
    pos = ref_ax.get_position()
    fig.set_layout_engine("none")
    cx0, _, cw, _ = cbar.ax.get_position().bounds
    cbar.ax.set_position([cx0, pos.y0, cw, pos.height])


def row_label(ax, text, pad=0.06):
    """Name a row of panels, left of its leftmost axes (no title anywhere in the figure)."""
    ax.text(-pad, 0.5, text, transform=ax.transAxes, rotation=90,
            va="center", ha="center", fontsize=PLOT.label_size)


def panel_tag(ax, text, loc=(0.025, 0.975)):
    """A caption-referable `(a)` / `(b)` marker inside the panel."""
    ax.text(*loc, text, transform=ax.transAxes, va="top", ha="left",
            fontsize=PLOT.annotation_size, zorder=6,
            bbox=dict(boxstyle="round,pad=0.18", facecolor="white",
                      edgecolor="0.7", linewidth=PLOT.axes_line_width, alpha=0.85))


def draw_graph_edges(ax, geo, color="0.45", alpha=0.7):
    """The backbone's message-passing graph (d <= 100 km) as faint segments."""
    i, j = np.nonzero(np.triu(geo["dist"] > 0.0) & (geo["dist"] <= geo["graph_max_dist"]))
    segs = np.stack([np.stack([geo["lon"][i], geo["lat"][i]], -1),
                     np.stack([geo["lon"][j], geo["lat"][j]], -1)], axis=1)
    ax.add_collection(LineCollection(segs, colors=color, alpha=alpha, zorder=2,
                                     linewidths=PLOT.axes_line_width, **TRANSFORM))
    return int(i.size)


def station_map(ax, geo, values, cmap=CMAP_T, vmin=None, vmax=None, size=18,
                mark_isolated=False, xlabels=False, ylabels=False):
    """Node-colored station scatter; NaN nodes drawn hollow.

    `mark_isolated` rings the stations with no neighbour inside the message-passing
    cutoff. It is off by default and on only in the network panel (§6.7): being isolated
    is a property of *that* graph, and a ring on a panel where the graph is not drawn
    reads as a marked data point rather than as a marked connection.
    """
    v = np.asarray(values, dtype=np.float64)
    bad = ~np.isfinite(v)
    map_axes(ax, geo, xlabels=xlabels, ylabels=ylabels)
    if bad.any():
        ax.scatter(geo["lon"][bad], geo["lat"][bad], s=size, facecolors="none",
                   edgecolors="0.45", linewidths=0.4, zorder=3, **TRANSFORM)
    sc = ax.scatter(geo["lon"][~bad], geo["lat"][~bad], c=v[~bad], s=size, cmap=cmap,
                    vmin=vmin, vmax=vmax, edgecolors="0.25", linewidths=0.25, zorder=4,
                    **TRANSFORM)
    if mark_isolated:
        iso = geo["isolated"] & ~bad
        if iso.any():
            ax.scatter(geo["lon"][iso], geo["lat"][iso], s=size * 3.0, facecolors="none",
                       edgecolors="k", linewidths=0.5, zorder=5, **TRANSFORM)
    return sc


def edge_cut(corr, quantile=RHO_QUANTILE):
    """The |rho| above which a case's top `1 - quantile` station pairs sit.

    A *per-case* cut, because Sigma is predicted per case: on a day whose spectrum is
    flat, 75 % of the 6555 pairs clear any fixed cut and the map turns into a solid
    block of edges. Selecting a fixed *fraction* of pairs instead keeps every panel
    equally readable and moves the day-to-day change in correlation strength onto the
    shared grey scale, where it belongs.
    """
    rho = np.abs(off_diagonal(corr))
    return float(np.quantile(rho, quantile)) if rho.size else np.inf


def draw_correlation_edges(ax, geo, corr, threshold, norm=None, cmap=CMAP_EDGE,
                           lw_max=1.6, alpha=0.9):
    """Station pairs with |rho| >= `threshold`, drawn dark in proportion to |rho|.

    Grey rather than a signed colormap: the point of the panel is *which pairs the
    predictive ties together and how strongly*, and grey leaves the temperature colors
    of the nodes uncontested. The sign is checked and reported by the caller, since
    grey cannot show it -- for a Matern spectrum on the graph Laplacian every
    correlation is positive, and a negative edge would be news.

    `norm` is the shared `(lo, hi)` grey scale of the whole figure, so panels with
    different per-case cuts stay comparable; it defaults to this panel's own range.

    Returns `(collection, n_drawn, n_pairs, rho_drawn)`; strong edges are drawn last so
    they are not hidden by weak ones.
    """
    n = corr.shape[0]
    iu = np.triu_indices(n, k=1)
    rho = np.asarray(corr, dtype=np.float64)[iu]
    keep = np.isfinite(rho) & (np.abs(rho) >= threshold)
    i, j, rho_keep = iu[0][keep], iu[1][keep], rho[keep]
    order = np.argsort(np.abs(rho_keep))
    segs = np.stack([np.stack([geo["lon"][i], geo["lat"][i]], -1),
                     np.stack([geo["lon"][j], geo["lat"][j]], -1)], axis=1)[order]
    strength = np.abs(rho_keep[order])
    lo, hi = (threshold, float(np.abs(rho[np.isfinite(rho)]).max())) if norm is None else norm
    hi = max(hi, lo + 1e-6)
    lc = LineCollection(segs, cmap=cmap, norm=plt.Normalize(lo, hi),
                        linewidths=lw_max * strength / hi,
                        alpha=alpha, zorder=3, **TRANSFORM)
    lc.set_array(strength)
    ax.add_collection(lc)
    return lc, int(keep.sum()), n * (n - 1) // 2, rho_keep


def clip_to(values, vmax):
    """Clip for display only, so one outlier station cannot flatten a whole panel."""
    return np.clip(values, -vmax, vmax)

## 5 Load the runs

Two extractors per arm, both returning plain dicts of numpy arrays so the plotting code below
never touches a model:

* `analyze_case` — one forecast day: `obs`, `mu`, `sigma`, the dense `cov`/`corr`, and `N_DRAWS`
  drawn fields. It also runs two cheap self-tests and returns their results, so a corrupted
  rebuild is caught here rather than in a figure: the recovered `diag(Σ)` must equal
  `get_marg_std²`, and the smallest eigenvalue of Σ must not be meaningfully negative (the
  SB's rank-99 sample covariance sits at 0 by construction, not by failure).
* `scan_test_set` — the whole 730-day test split: the residual and σ matrices (NaNs preserved),
  the pairwise-complete empirical second moments, and the per-case Mahalanobis norm with its dof.
  ~45 s per arm on one GPU, so all five together are a few minutes.

`get_case` memoizes `analyze_case` on `(arm, case_index, n_draws)`, which is what lets every figure
below take a forecast case as a plain number: asking for a case that has not been seen costs one
forward pass per arm and nothing thereafter. The eager `cases` / `multi` / `scans` below are the
defaults the summary and the multi-case grid use; a figure called with a different `case_index`
fills the cache itself.

`has_cov` is `False` exactly for the deterministic arm (Σ ≡ 0, so σ ≡ 0, ρ undefined, every draw
equals μ and no Cholesky exists). It is set from the data, not from the arm name, and the panels
below branch on it.

In [ ]:
@torch.no_grad()
def analyze_case(name, run_dir, case_index=CASE_INDEX, n_draws=N_DRAWS, device=DEVICE):
    """Everything the single-case panels need, for one arm and one forecast day."""
    model, method, dm, cfg, ckpt = build_model_from_run(run_dir, device=device)
    x, y = dm.test_dataset[case_index]
    x = torch.as_tensor(x).unsqueeze(0).to(device)
    y = torch.as_tensor(y).unsqueeze(0).to(device)

    pred = model(x)
    mu = method.get_mean(pred)[0, 0].cpu().numpy()
    sigma = method.get_marg_std(pred)[0, 0].cpu().numpy()
    cov = full_covariance(method, pred)[0].double().cpu().numpy()
    obs = y[0, 0].cpu().numpy()
    has_cov = float(np.max(sigma)) > 0.0
    # A Dirac's `get_samples` returns two copies of mu whatever `n_samples` says, so
    # there is nothing to draw for the deterministic arm.
    samples = (method.get_samples(pred, n_samples=n_draws)[0, 0].cpu().numpy()
               if has_cov else None)

    # Self-tests (reported, not asserted, so a bad arm still plots).
    diag_err = float(np.abs(np.diag(cov) - sigma ** 2).max()
                     / max(float(sigma.max()) ** 2, 1e-12))
    ev = np.linalg.eigvalsh(cov)
    return {
        "name": name, "run_dir": str(run_dir), "ckpt": ckpt,
        "method_cls": type(method).__name__,
        "loss": cfg.method.get("loss"), "stage1_loss": cfg.method.get("stage1_loss"),
        "case_index": case_index,
        "date": pd.to_datetime(dm.meta["test_time"][case_index]),
        "obs": obs, "mask": np.isfinite(obs),
        "mu": mu, "sigma": sigma, "cov": cov, "corr": cov_to_corr(cov),
        "samples": samples, "has_cov": has_cov,
        "diag_rel_err": diag_err,
        "eig_min": float(ev.min()), "eig_max": float(ev.max()),
        "n_members": int(x.shape[2]),
    }


@torch.no_grad()
def scan_test_set(name, run_dir, max_batches=MAX_TEST_BATCHES, device=DEVICE):
    """Test-split diagnostics for one arm: residuals, sigmas, empirical moments, Mahalanobis."""
    model, method, dm, cfg, ckpt = build_model_from_run(run_dir, device=device)
    resid, sd_all, maha, dof = [], [], [], []
    count = 0

    for bi, (x, y) in enumerate(dm.test_dataloader()):
        if max_batches is not None and bi >= max_batches:
            break
        x, y = x.to(device), y.to(device)
        pred = model(x)
        mu = method.get_mean(pred)[:, 0].double().cpu().numpy()
        sd = method.get_marg_std(pred)[:, 0].double().cpu().numpy()
        r = y[:, 0].double().cpu().numpy() - mu                  # NaN where unreported
        has_cov = float(sd.max()) > 0.0
        if has_cov:
            cov = full_covariance(method, pred).double().cpu().numpy()
            for b in range(cov.shape[0]):
                m, d = masked_maha(cov[b], r[b])
                maha.append(m)
                dof.append(d)
        else:
            for b in range(r.shape[0]):
                maha.append(np.nan)
                dof.append(int(np.isfinite(r[b]).sum()))
        resid.append(r)
        sd_all.append(sd)
        count += r.shape[0]

    resid = np.concatenate(resid, axis=0)
    sd_all = np.concatenate(sd_all, axis=0)
    emp_S, pair_count = pairwise_complete_stats(resid)
    return {
        "name": name, "n_cases": count,
        "resid": resid, "sigma": sd_all,
        "has_cov": float(sd_all.max()) > 0.0,
        "n_obs": np.isfinite(resid).sum(1),
        "emp_S": emp_S, "emp_corr": cov_to_corr(emp_S), "pair_count": pair_count,
        "maha": np.array(maha), "dof": np.array(dof, dtype=float),
    }

_CASE_CACHE = {}


def get_case(name, case_index=CASE_INDEX, n_draws=N_DRAWS):
    """`analyze_case` for one arm and one forecast day, memoized on the three keys.

    This is what makes a forecast case a *number* everywhere below: a figure takes
    `case_index`, asks for the arms it draws, and pays one forward pass per arm the first
    time that day is requested. Nothing else in the notebook builds a case.
    """
    key = (name, int(case_index), int(n_draws))
    if key not in _CASE_CACHE:
        _CASE_CACHE[key] = analyze_case(name, RUNS[name], case_index=int(case_index),
                                        n_draws=int(n_draws))
    return _CASE_CACHE[key]


def get_cases(case_index=CASE_INDEX, arms=None, n_draws=N_DRAWS):
    """`{arm: case}` for one forecast day, in the order `arms` gives (default: all of `RUNS`)."""
    return {a: get_case(a, case_index, n_draws) for a in (arms if arms is not None else RUNS)}

In [ ]:
_, _, DM, _, _ = build_model_from_run(next(iter(RUNS.values())))
GEO = station_geometry(DM)

print(f"{GEO['dist'].shape[0]} stations | {GEO['n_edges']} directed edges at "
      f"<= {GEO['graph_max_dist']:.0f} km | degree min/median/max = "
      f"{GEO['degree'].min()}/{int(np.median(GEO['degree']))}/{GEO['degree'].max()}")
print(f"isolated stations: {int(GEO['isolated'].sum())} "
      f"({', '.join(str(s) for s in GEO['sid'][GEO['isolated']])})")
print(f"domain: {GEO['lat'].min():.2f}-{GEO['lat'].max():.2f} degN, "
      f"{GEO['lon'].min():.2f}-{GEO['lon'].max():.2f} degE")

# The reference case for every arm, the multi-case grid's cases for the two arms that grid
# shows, and a full test-set scan per arm. Everything goes through `get_case`, so a figure
# asked for one of these days later reuses the forward pass rather than repeating it.
cases = get_cases(CASE_INDEX)
multi = {name: {ci: get_case(name, ci) for ci in CASE_INDICES}
         for name in (LORA, CHOLESKY, GENERATIVE, OURS)}
scans = {name: scan_test_set(name, rd) for name, rd in RUNS.items()}

In [ ]:
ref = cases[OURS]
print(f"reference case {CASE_INDEX}: {ref['date']:%Y-%m-%d} | "
      f"{int(ref['mask'].sum())}/{ref['mask'].size} stations reported | "
      f"{ref['n_members']} ensemble members\n")

for name, c in cases.items():
    s = scans[name]
    ok = np.isfinite(s["maha"])
    rho = off_diagonal(c["corr"])
    scores = load_metrics(c["run_dir"])
    print(f"{LABELS[name]:<14s} {c['method_cls']:<20s} loss={str(c['loss']):<13s} "
          f"stage1={str(c['stage1_loss'])}")
    print(f"{'':14s} ckpt {c['ckpt']}")
    print(f"{'':14s} diag rel err {c['diag_rel_err']:.2e} | "
          f"eig(Sigma) in [{c['eig_min']:.3e}, {c['eig_max']:.3e}] | "
          f"max off-diag |rho| {np.abs(rho).max() if rho.size else float('nan'):.3f}")
    print(f"{'':14s} scan: {s['n_cases']} cases, "
          f"{np.isfinite(s['resid']).mean() * 100:.2f}% observed | "
          f"maha/dof {np.nanmean(s['maha'][ok] / s['dof'][ok]) if ok.any() else float('nan'):.3f} "
          f"({int((~ok).sum())} without a Cholesky)")
    if scores is not None:
        print(f"{'':14s} test: " + "  ".join(
            f"{k}={v:.4f}" for k, v in scores.items() if v is not None))

# The edge cuts of Fig. 1, one per arm (every arm either figure can show, including Full
# cov. for the appendix): the arms differ by a factor of two in correlation strength, so a
# single cut would draw every pair of one arm and none of another. Each panel draws the same
# *fraction* of pairs and the shared grey scale carries the difference in strength -- the
# same convention the multi-case grid uses across days.
print(f"\ncorrelation-edge cuts on case {EDGE_CASE} "
      f"({'per-arm ' + str(RHO_QUANTILE) + ' quantile' if RHO_THRESHOLD is None else 'fixed'}):")
for name in EDGE_ARMS_APPENDIX:
    _rho = np.abs(off_diagonal(get_case(name, EDGE_CASE)["corr"]))
    _cut = edge_cut(get_case(name, EDGE_CASE)["corr"]) if RHO_THRESHOLD is None else RHO_THRESHOLD
    print(f"{'':2s}{LABELS[name]:<14s} |rho| >= {_cut:.3f} -> {int((_rho >= _cut).sum())} "
          f"of {_rho.size} pairs (max |rho| {_rho.max():.3f})")

### 6.1 The realized field and the correlation structure each arm predicts for it

**Fig. 1.** Panel (a) is the observed 2-m temperature on the chosen day — the realization the
models are asked to describe, with hollow markers where a station did not report. Panels (b)–(c)
are the predictive mean of each arm with **the station pairs its own covariance ties together**
drawn on top: LorD (r = 1), then SCORE (graph Fourier transform, Matérn filter). One shared temperature scale, so μ and
the truth are directly comparable by eye, and one shared grey scale for |ρ|, so the two arms are
comparable *in strength* and not only in shape. Three panels fit the single-column width, which is
what the main-text figure below uses; §6.1's second figure is the double-column, three-case
appendix version with Chol added back in as its own column.

The edges are **not** the model's message-passing graph — those are the ≤ 100 km connections in
§6.7 — but the pairs whose predictive correlation is in the top `1 − RHO_QUANTILE` **of that arm**.
A per-arm cut rather than one absolute cut, for the same reason the multi-case grid uses one per
day: the arms differ by roughly a factor of two in predicted correlation, and any single cut draws
every pair of the strongest arm and none of the weakest. Each panel therefore draws the same 328
of 6555 pairs and the shared grey scale is where the difference in strength shows up.

Grey rather than a signed colormap: the nodes already own the color axis. The sign is what the
cell prints instead, per arm — for a Matérn spectrum on the graph Laplacian every correlation is
positive by construction, so a negative edge in panel (c) would be news. LorD carries no
such constraint (its correlations are `sign(u_i u_j)` for a single global loading vector `u`, so a
sign flip needs only one station with the opposite loading), but whether a pair strong enough to
clear the cut is negative is a per-case fact, and the printed count is what says so. Chol has
the same lack of constraint and is where it is more likely to show up — see the appendix figure.

SB is deliberately absent from the main-text figure: its Σ is a 100-member
sample estimate, so its off-diagonal is an estimate with its own sampling error rather than
a fitted parameter, and a threshold on it would select noise as much as structure -- exactly
why it is also absent from §6.4 and §6.5 wherever a *fitted* quantity is asked of it. The
appendix figure below adds it back as a fourth arm precisely to show that estimate: the same
`full_covariance` call the closed-form arms use returns its sample second moments (§3) for
this arm too, from the M members that forward pass already drew, so nothing new has to be
sampled for the edges it draws.

This is the one main-text figure whose panels are stretched towards the domain's true shape
(`MAIN_FIG_ASPECT`): PlateCarree measures both axes in degrees, which at 49.6 °N draws the domain
1.55× too wide. The grids in §6.2–§6.4, and the appendix version of this figure, stay in raw
degrees, where the same correction would add a page of height for no extra information.

In [ ]:
def plot_truth_and_edges(case_index=EDGE_CASE, geo=None, arms=None, columns="single",
                         aspect_scale=MAIN_FIG_ASPECT, quantile=RHO_QUANTILE):
    """Fig. 1: the observed field, then each arm's mu with its own strongest correlation edges.

    `case_index` is a plain index into the test split; `arms` defaults to `EDGE_ARMS` (LoRa
    and Ours -- the two arms a single-column, three-panel figure has room for). See
    `plot_truth_and_edges_grid` below for the appendix version: Full cov. added back in as a
    column, three forecast cases as rows.
    """
    geo = GEO if geo is None else geo
    arms = list(EDGE_ARMS if arms is None else arms)
    cs = get_cases(case_index, arms)
    obs = cs[arms[0]]["obs"]

    pool = np.concatenate([obs[np.isfinite(obs)]] + [cs[a]["mu"] for a in arms])
    vmin, vmax = float(pool.min()), float(pool.max())

    # One cut per arm (the same fraction of pairs everywhere) and one grey scale for the whole
    # figure, from the weakest cut to the strongest pair any arm predicts -- so the panels are
    # equally readable and the difference in correlation *strength* stays visible.
    cuts = {a: (edge_cut(cs[a]["corr"], quantile) if RHO_THRESHOLD is None else RHO_THRESHOLD)
            for a in arms}
    rho_norm = (min(cuts.values()),
                max(float(np.abs(off_diagonal(cs[a]["corr"])).max()) for a in arms))

    fig, axes = map_grid(geo, 1, len(arms) + 1, columns=columns, extra_h=0.30,
                         aspect_scale=aspect_scale)
    sc = station_map(axes[0, 0], geo, obs, cmap=CMAP_T, vmin=vmin, vmax=vmax, size=8)
    stretch_domain(axes[0, 0], aspect_scale)
    panel_tag(axes[0, 0], r"(a) $y$")
    for j, name in enumerate(arms, start=1):
        station_map(axes[0, j], geo, cs[name]["mu"], cmap=CMAP_T, vmin=vmin, vmax=vmax, size=8)
        stretch_domain(axes[0, j], aspect_scale)
        lc, drawn, total, rho_drawn = draw_correlation_edges(
            axes[0, j], geo, cs[name]["corr"], threshold=cuts[name], norm=rho_norm)
        panel_tag(axes[0, j], f"({chr(ord('a') + j)}) {SHORT_LABELS[name]}")
        print(f"{LABELS[name]:<14s} {drawn:4d} of {total} pairs drawn "
              f"(|rho| >= {cuts[name]:.3f}, max {np.abs(off_diagonal(cs[name]['corr'])).max():.3f}), "
              f"{int((rho_drawn < 0).sum())} of them negative")

    cb = fig.colorbar(sc, ax=axes[0, :], location="bottom", fraction=0.05, aspect=45, pad=0.02)
    cb.set_label("2\\,m temperature [$^\\circ$C]", size=PLOT.colorbar_label_size)
    cbe = fig.colorbar(lc, ax=axes[0, :], location="right", fraction=0.0285, pad=0.02,
                       aspect=20)
    cbe.set_label(r"$|\rho_{ij}|$", size=PLOT.colorbar_label_size)
    # `location="right"` sizes to the row's nominal grid cell, not to what the fixed-aspect
    # panels actually occupy within it once `stretch_domain` has shrunk them -- fix the bar
    # up to match (see `sync_colorbar_height`).
    sync_colorbar_height(fig, cbe, axes[0, -1])
    c0 = cs[arms[0]]
    print(f"{c0['date']:%Y-%m-%d} (case {case_index}): "
          f"{int(c0['mask'].sum())}/{c0['mask'].size} stations reported")
    return fig


fig = plot_truth_and_edges(EDGE_CASE)
savefig(fig, "euppbench_truth_correlation")
plt.show()

#### Appendix: Fig. 1 with every arm and more forecast cases

**Fig. 1 (appendix).** The double-column, three-row version of the figure above, with every arm
back in: panels (b)-(e) are now LorD (r = 1), Chol, SB's *sample*
correlation, and SCORE -- `DRAW_ARMS`' order, since this is the first
figure to put all four covariance-carrying arms side by side. Each row is a different forecast
day -- 2017-01-01 and 2018-01-01 (both winters) and the 2017-07-02 summer case between them, the
first three of `CASE_INDICES` -- so the correlation structure each arm predicts is seen changing
case to case rather than fixed to one day.

One shared temperature scale and one shared grey |ρ| scale across the whole grid, exactly as in
the main figure, only now pooled over three cases and four arms: twelve edge cuts (three cases ×
four arms), not four, feed the one scale. The colorbars are both horizontal here -- temperature
below the grid, |ρ| above the arm columns -- rather than the main figure's vertical bar on the
right: `sync_colorbar_height` fixes a bar to *one* row's box, and at three rows there is no single
box to fix it to.

The SB column is the odd one out in the same way it always is: panels (b), (c) and (e)
draw a *fitted* correlation, so a strong edge is a claim the model made; panel (d) draws the
*sample* correlation of M = 100 forward-pass members, so a strong edge there is only as reliable
as that many draws make it, and the per-case pair count printed alongside the others is not
evidence against that -- it just says how many pairs happen to clear the cut, not how trustworthy
each one is.

In [ ]:
def plot_truth_and_edges_grid(case_indices=EDGE_CASES_APPENDIX, geo=None, arms=None,
                              columns="double", quantile=RHO_QUANTILE):
    """Fig. 1 (appendix): rows = forecast cases, columns = obs + each arm's mu with its edges.

    The same panels as `plot_truth_and_edges`, with `EDGE_ARMS_APPENDIX` (Full cov. and the
    generative arm included) and one row per case in `case_indices` instead of one case. Left
    in raw-degree axes rather than `stretch_domain`'s true-shape stretch, like the other
    multi-row grids -- at three rows
    the correction would cost a full page of height (see §6.1's closing note).
    """
    geo = GEO if geo is None else geo
    arms = list(EDGE_ARMS_APPENDIX if arms is None else arms)
    rows = [get_cases(ci, arms) for ci in case_indices]

    pool = np.concatenate(
        [rows[0][arms[0]]["obs"][np.isfinite(rows[0][arms[0]]["obs"])]]
        + [r[a]["mu"] for r in rows for a in arms])
    vmin, vmax = float(pool.min()), float(pool.max())

    # One cut per (case, arm) -- twelve here, not three -- and one grey scale for the whole
    # grid: the same convention as the main figure, extended across rows.
    cuts = [{a: (edge_cut(r[a]["corr"], quantile) if RHO_THRESHOLD is None else RHO_THRESHOLD)
             for a in arms} for r in rows]
    rho_norm = (min(c for cd in cuts for c in cd.values()),
                max(float(np.abs(off_diagonal(r[a]["corr"])).max())
                    for r, cd in zip(rows, cuts) for a in arms))

    fig, axes = map_grid(geo, len(case_indices), len(arms) + 1, columns=columns, extra_h=0.58)
    for i, (ci, r, cd) in enumerate(zip(case_indices, rows, cuts)):
        obs = r[arms[0]]["obs"]
        sc = station_map(axes[i, 0], geo, obs, cmap=CMAP_T, vmin=vmin, vmax=vmax, size=10)
        row_label(axes[i, 0], f"{r[arms[0]]['date']:%Y-%m-%d}")
        for j, name in enumerate(arms, start=1):
            station_map(axes[i, j], geo, r[name]["mu"], cmap=CMAP_T, vmin=vmin, vmax=vmax,
                        size=10)
            lc, drawn, total, rho_drawn = draw_correlation_edges(
                axes[i, j], geo, r[name]["corr"], threshold=cd[name], norm=rho_norm)
            print(f"{r[name]['date']:%Y-%m-%d} {LABELS[name]:<14s} {drawn:4d} of {total} pairs "
                  f"drawn (|rho| >= {cd[name]:.3f}), {int((rho_drawn < 0).sum())} negative")
        if i == 0:
            panel_tag(axes[0, 0], r"(a) $y$")
            for j, name in enumerate(arms, start=1):
                panel_tag(axes[0, j], f"({chr(ord('a') + j)}) $\\mu$, {LABELS[name]}")

    cb = fig.colorbar(sc, ax=axes[:, :], location="bottom", fraction=0.025, pad=0.01, aspect=55)
    cb.set_label("2\\,m temperature [$^\\circ$C]", size=PLOT.colorbar_label_size)
    cbe = fig.colorbar(lc, ax=axes[:, 1:], location="top", fraction=0.025, pad=0.01, aspect=42)
    cbe.set_label(r"$|\rho_{ij}|$", size=PLOT.colorbar_label_size)
    return fig


fig = plot_truth_and_edges_grid()
savefig(fig, "euppbench_truth_correlation_appendix")
plt.show()


## 7 Overview figure -- ETTh2 and EUPPBench, double column

This is Figure 3 of the paper.

Fig. 1's three panels next to their time-series counterpart, drawn as **one** double-column
figure so the six panels share one row and the two colorbars one baseline -- which two
separately saved figures set side by side in LaTeX cannot guarantee. Saved as
`overview_etth2_euppbench.pdf`: each data set on a grey card, as in the theory overview, and
every panel labelled by name only, centered above it. `plot_overview(ts, boxed=False)` draws
the plain alternative -- no cards, the two data sets tagged "(a)" / "(b)" -- which is not saved.

The panels, left to right:

* **ETTh2**, univariate, $T = 96$, seed `SEED`: one test window of SCORE -- the last 72 h
  of input, the truth, the predictive mean (dashed, drawn over the truth) and the
  $\pm 2\sigma$ marginal band -- then per arm the correlation over the horizon split along the
  diagonal, the arm's **test-set mean** $\overline{\Sigma}$ as a correlation above it and the
  empirical correlation of that arm's own residuals below it. The same numbers as
  `analyze_ts_predictions_univariate.ipynb` section 8, which this section re-derives rather
  than reads, so it runs on its own.
* **EUPPBench**: Fig. 1 unchanged in content -- the observed field, then each arm's $\mu$
  with its own strongest correlation edges (the per-arm `RHO_QUANTILE` cut, one grey scale).

Which triangle is which is said once, outside the matrices, where it cannot cover them:
"Model" left of the first one (its left edge borders the model triangle) and "Empirical" right
of the last (its right edge borders the empirical one). `OVERVIEW_TRIANGLE_LABELS` switches to
labels beside every matrix (`"outside"`) or inside the triangles (`"inside"`).

Both halves compare LorD $r{=}1$ with SCORE, both kernel-score fitted. The correlation
colormap is sequential (reversed crameri `lipari`, dark = high) because the diverging
red-blue is EUPPBench's temperature scale. The dark top-left corner of SCORE's correlation
panel is the circulant core's wrap-around -- $\rho(1, 96) = \rho(1, 2)$ exactly, against much
less in the residuals (the next cell prints both).

Every panel is `OVERVIEW_PANEL_H` high: the correlation panels are squares, the maps keep the
domain shape `MAIN_FIG_ASPECT` gives them, and the forecast panel takes the width that is
left. `OVERVIEW_BOXED` switches the default of `plot_overview` between the two versions.
In the boxed version every card leaves `CARD_PAD` between its border and the nearest thing
inside it -- panel, tick label, title or colorbar label -- on all four sides, and the cards sit
`CARD_SPLIT` apart. The margins are measured off a first drawn placement rather than set by
hand, so they stay even when a font, a tick or a label changes.

In [ ]:
from src.evaluation.calibration import build_model_from_run as build_calibration_run

#: The time-series half: ETTh2, univariate, T = 96, the same seed as the EUPPBench arms.
from src.utils.paths import configured_path
TS_ROOT = configured_path("results_dir") / "time_series/etth2/univariate/T96"
TS_DATA_PATH = str(configured_path("time_series_dir") / "ETT-small/ETTh2.csv")
TS_ARMS = {LORA: "phys_lora_r1_kernel_score", OURS: "whitened_spectral_diag_s1crps_kernel_score"}
TS_WINDOW = 400          # test window of panel (a)
TS_CADENCE_HOURS = {"ett_hour": 1.0, "ett_minute": 0.25}


@torch.no_grad()
def ts_extract(slug, window=TS_WINDOW, seed=SEED):
    """One ETTh2 arm: the `window` forecast in degC and the test-set correlation pair.

    `model_corr` is the correlation of the test-set mean predictive covariance, `emp_corr`
    that of the arm's own residuals -- the pair the split panels draw.
    """
    model, method, dm, cfg, ckpt = build_calibration_run(
        TS_ROOT / slug / f"seed{seed}", device=DEVICE, data_path=TS_DATA_PATH,
        num_workers=0, eval_batch_size=64, persistent_workers=False)
    ds = dm.test_dataset
    scale, shift = float(ds.scaler.scale_[-1]), float(ds.scaler.mean_[-1])
    x, y = ds[window]
    pred = model(x.unsqueeze(0).to(DEVICE))
    out = {
        "ckpt": ckpt,
        "step_hours": TS_CADENCE_HOURS[str(cfg.datamodule.data_split)],
        "context": x[0].cpu().numpy() * scale + shift,
        "target": y[0].cpu().numpy() * scale + shift,
        "mean": method.get_mean(pred)[0, 0].cpu().numpy() * scale + shift,
        "std": method.get_marg_std(pred)[0, 0].cpu().numpy() * scale,
    }
    cov_sum, resid, n = 0.0, [], 0
    for xb, yb in dm.test_dataloader():
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        p = model(xb)
        cov_sum = cov_sum + full_covariance(method, p).double().sum(0)
        resid.append((yb.flatten(start_dim=1) - method.get_mean(p).flatten(start_dim=1)).cpu())
        n += xb.shape[0]
    out["n"] = n
    out["model_corr"] = cov_to_corr((cov_sum / n).cpu().numpy())
    out["emp_corr"] = cov_to_corr(np.cov(torch.cat(resid).double().numpy(), rowvar=False))
    return out


ts = {name: ts_extract(slug) for name, slug in TS_ARMS.items()}
for name, t in ts.items():
    m, e = t["model_corr"], t["emp_corr"]
    print(f"{LABELS[name]:<12s} n={t['n']}  rho(1,2): model {m[0, 1]:.2f} / empirical "
          f"{e[0, 1]:.2f}   rho(1,{m.shape[0]}): model {m[0, -1]:.2f} / empirical {e[0, -1]:.2f}")

In [ ]:
from matplotlib import ticker as mticker
from matplotlib.layout_engine import PlaceHolderLayoutEngine
from matplotlib.patches import FancyBboxPatch
from matplotlib.transforms import Bbox

#: (ETTh2)'s correlation colormap. Sequential, because the diverging red-blue is the EUPPBench
#: temperature scale and every correlation drawn there is positive; reversed `lipari` is
#: crameri's magma with dark = high.
try:
    from cmap import Colormap
    CMAP_CORR = Colormap("crameri:lipari").to_mpl().reversed()
except Exception:                     # `cmap` is optional in plot_config as well
    CMAP_CORR = plt.get_cmap("magma_r")
#: Height of every panel, inches. The correlation panels are squares of this side, the maps
#: keep Fig. 1's domain shape, and the forecast panel takes the width that leaves.
OVERVIEW_PANEL_H = 0.78
OVERVIEW_CONTEXT_H = 72     # hours of input drawn left of the forecast origin
#: Grey cards behind the two halves, styled as in theoretical_visualizations.ipynb's
#: `draw_flowchart(card=...)`. The boxed figure is the paper's; `boxed=False` is the plain
#: alternative (with "(a)" / "(b)"), not saved.
OVERVIEW_BOXED = True
CARD_FC, CARD_EC, CARD_ROUND = "#f2f2f0", "#e2e2de", 0.08
#: Inches: from a card's border to the nearest thing inside it -- panel, tick label, title or
#: colorbar label -- on all four sides of both cards; and between the two cards.
CARD_PAD, CARD_SPLIT = 0.09, 0.14
#: Where the correlation panels say which triangle is which: "inside" writes "Model" /
#: "Empirical" into the two triangles; "outside" puts them next to each panel instead, "Model"
#: along the left edge and "Empirical" along the right one (those edges bound exactly those
#: triangles); "once" does that only left of the first panel and right of the last.
OVERVIEW_TRIANGLE_LABELS = "once"


def split_triangle(upper, lower):
    """`upper` above the diagonal and `lower` below it, as drawn with `origin="lower"`."""
    i, j = np.indices(upper.shape)
    return np.where(i > j, upper, np.where(i < j, lower, 1.0))


def ink_on(cmap, value):
    """Black or white, whichever reads on `cmap`'s color at `value` (0..1)."""
    r, g, b, _ = cmap(float(value))
    return "0.15" if 0.2126 * r + 0.7152 * g + 0.0722 * b > 0.45 else "white"


def overview_title(ax, text, loc="left"):
    """A label above the panel, no box: the dataset tags go left, the method names center."""
    ax.set_title(text, loc=loc, fontsize=PLOT.annotation_size, pad=2)


def plot_overview(ts, case_index=EDGE_CASE, arms=None, geo=None, quantile=RHO_QUANTILE,
                  aspect_scale=MAIN_FIG_ASPECT, panel_h=OVERVIEW_PANEL_H,
                  context_h=OVERVIEW_CONTEXT_H, boxed=OVERVIEW_BOXED, letters=None,
                  triangle_labels=OVERVIEW_TRIANGLE_LABELS):
    """ETTh2: forecast and split correlations; EUPPBench: Fig. 1 -- one row.

    Placed in inches rather than by constrained layout: the map panels are fixed-aspect
    GeoAxes, and constrained layout sizes their grid cells rather than their drawn boxes,
    so neither one shared panel height nor colorbars exactly as wide as their panels would
    hold. The map rectangles take the domain's shape, so the GeoAxes fill them exactly.

    The margins around each half are *measured*, not guessed: a first placement is drawn,
    each half's labels are found to reach so far past its panels and colorbars, and the
    final placement leaves exactly `CARD_PAD` beyond that on every side of both cards (and
    the forecast panel absorbs whatever width this frees or costs).

    `boxed` puts each half on a grey card. `letters` tags the two data sets "(a)" / "(b)",
    left-aligned; by default only without the cards, which already separate the halves --
    otherwise every label is a name centered over its panel.
    """
    letters = (not boxed) if letters is None else letters
    def dataset_title(ax, tag, name):
        if letters:
            overview_title(ax, f"({tag}) {name}")
        else:
            overview_title(ax, name, loc="center")

    geo = GEO if geo is None else geo
    arms = list(EDGE_ARMS if arms is None else arms)
    width = pc.width_of("double")
    extent = map_extent(geo)
    h = side = panel_h                                                   # correlation squares
    w_map = h / (aspect_scale * (extent[3] - extent[2]) / (extent[1] - extent[0]))
    gap_ab, gap = 0.17, 0.08                                             # inches
    gap_sq = gap                            # between the two correlation panels
    if triangle_labels != "inside":
        gap_ab = 0.20                       # "Model" left of the first panel
    if triangle_labels == "outside":
        gap_sq = 0.17                       # one panel's "Empirical", the next one's "Model"
    rbar_gap, rbar_w = 0.05, 0.053          # the |rho| bar right of the maps
    cbar_pad, cbar_h = 0.04, 0.056          # the two horizontal bars under the panels
    margin = 0.02                           # canvas edge to card (or to content, unboxed)
    pad, split = (CARD_PAD, CARD_SPLIT) if boxed else (0.0, 0.25)

    # A placeholder engine rather than layout="none": savefig restores a None engine by
    # re-reading rcParams, where `pc.apply_style` switched constrained layout on. Every axes
    # starts anywhere; `place` below puts them.
    fig = plt.figure(figsize=(width, 2.0), layout=PlaceHolderLayoutEngine(True, True))
    geo_kw = {"projection": PROJECTION} if HAS_CARTOPY else {}
    ax_a = fig.add_axes([0, 0, 0.1, 0.1])
    ax_ts = [fig.add_axes([0, 0, 0.1, 0.1]) for _ in arms]
    ax_d, ax_e, ax_f = [fig.add_axes([0, 0, 0.1, 0.1], **geo_kw) for _ in range(3)]
    cax_ts, cax_t, cax_rho = [fig.add_axes([0, 0, 0.1, 0.1]) for _ in range(3)]
    ax_a.set_facecolor("white")                          # stays white on the grey card

    # ETTh2: band, input, truth, and SCORE's mean dashed on top of the truth. The ticks carry the
    # unit, so there is no axis label.
    w = ts[OURS]
    color, dt = ARM_COLORS[OURS], w["step_hours"]
    T = len(w["target"])
    lead = np.arange(1, T + 1) * dt
    n_ctx = int(round(context_h / dt))
    ax_a.fill_between(lead, w["mean"] - 2 * w["std"], w["mean"] + 2 * w["std"], color=color,
                      alpha=PLOT.band_alpha, lw=0, zorder=1)
    ax_a.plot(np.arange(-n_ctx + 1, 1) * dt, w["context"][-n_ctx:],
              color=pc.COLORS["guide"], lw=0.8, zorder=2)
    ax_a.plot(lead, w["target"], color="k", lw=0.8, zorder=3)
    ax_a.plot(lead, w["mean"], color=color, lw=1.0, ls=(0, (3, 1.5)), zorder=4)
    ax_a.axvline(0, color=pc.COLORS["guide"], lw=0.5, ls=":", zorder=0)
    ax_a.set_xlim(-n_ctx * dt, T * dt)
    hours = [-48, 0, 48, 96]
    ax_a.set_xticks(hours, labels=[f"${hr:d}$h" for hr in hours])
    dataset_title(ax_a, "a", "ETTh2")

    # Then per arm: model above the diagonal, the arm's own residuals below it, on one scale
    # that starts at 0 unless some correlation is negative.
    imgs = [split_triangle(ts[a]["model_corr"], ts[a]["emp_corr"]) for a in arms]
    vmin = min(0.0, np.floor(10 * min(np.nanmin(im) for im in imgs)) / 10)
    norm = plt.Normalize(vmin, 1.0)
    extra = {}                              # text outside an axes, for `measure` below
    for k, (ax, a, img) in enumerate(zip(ax_ts, arms, imgs)):
        n = img.shape[0]
        im = ax.imshow(img, origin="lower", cmap=CMAP_CORR, norm=norm, aspect="auto",
                       extent=(0.5, n + 0.5, 0.5, n + 0.5), interpolation="nearest",
                       rasterized=True)
        ax.plot([0.5, n + 0.5], [0.5, n + 0.5], color=ink_on(CMAP_CORR, 1.0), lw=0.3)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
        overview_title(ax, SHORT_LABELS[a], loc="center")
        if triangle_labels == "inside":
            # Out on the anti-diagonal, away from the dark band along the diagonal: there the
            # lag is ~T/2, the weakest correlation even with the circulant wrap-around. The ink
            # follows the pixels right under each word.
            for x, y, text in [(0.24, 0.76, "Model"), (0.76, 0.24, "Empirical")]:
                r, c = int(y * n), int(x * n)
                local = img[max(r - 6, 0):r + 6, max(c - 6, 0):c + 6]
                ax.text(x, y, text, transform=ax.transAxes, rotation=45, ha="center",
                        va="center", fontsize=PLOT.annotation_size - 1,
                        color=ink_on(CMAP_CORR, norm(np.nanmedian(local))))
        else:
            # The whole left edge borders the upper (model) triangle and the whole right edge
            # the lower (empirical) one; each word runs from the corner where its triangle is
            # widest, so in the gap between two panels one hangs from the top and the other
            # stands on the bottom, and they do not meet. "Model" reads upwards like a y-label,
            # "Empirical" downwards like a label on a right-hand axis.
            sides = [(0.0, 1.0, "Model", "right", "top", -2, 90),
                     (1.0, 0.0, "Empirical", "left", "bottom", 2, 270)]
            if triangle_labels == "once":
                sides = [sd for sd, keep in zip(sides, (k == 0, k == len(arms) - 1)) if keep]
            for x, y, text, ha, va, dx, rot in sides:
                extra.setdefault(ax, []).append(ax.annotate(
                    text, (x, y), xycoords="axes fraction", xytext=(dx, 0),
                    textcoords="offset points", rotation=rot, ha=ha, va=va,
                    fontsize=PLOT.annotation_size - 1, color="0.25"))
    cb = fig.colorbar(im, cax=cax_ts, orientation="horizontal",
                      format=mticker.FuncFormatter(lambda v, _: f"${v:g}$"))   # math minus
    cb.set_ticks(np.round(np.arange(vmin, 1.0 + 1e-9, 0.2), 1))
    cb.set_label(r"Correlation $\rho$", size=PLOT.colorbar_label_size)

    # EUPPBench: Fig. 1's panels -- the observed field, then each arm's mu with its own
    # strongest correlation edges on one shared grey scale (see `plot_truth_and_edges`).
    cs = get_cases(case_index, arms)
    obs = cs[arms[0]]["obs"]
    pool = np.concatenate([obs[np.isfinite(obs)]] + [cs[a]["mu"] for a in arms])
    vmin_t, vmax_t = float(pool.min()), float(pool.max())
    cuts = {a: (edge_cut(cs[a]["corr"], quantile) if RHO_THRESHOLD is None else RHO_THRESHOLD)
            for a in arms}
    rho_norm = (min(cuts.values()),
                max(float(np.abs(off_diagonal(cs[a]["corr"])).max()) for a in arms))
    sc = station_map(ax_d, geo, obs, cmap=CMAP_T, vmin=vmin_t, vmax=vmax_t, size=8)
    stretch_domain(ax_d, aspect_scale)
    # With its letter, left-aligned and wider than its 0.71 in panel; the method names beside
    # it are centered, so it clears them.
    dataset_title(ax_d, "b", "EUPPBench")
    for ax, a in zip([ax_e, ax_f], arms):
        station_map(ax, geo, cs[a]["mu"], cmap=CMAP_T, vmin=vmin_t, vmax=vmax_t, size=8)
        stretch_domain(ax, aspect_scale)
        lc, drawn, total, rho_drawn = draw_correlation_edges(
            ax, geo, cs[a]["corr"], threshold=cuts[a], norm=rho_norm)
        overview_title(ax, SHORT_LABELS[a], loc="center")
        print(f"{LABELS[a]:<14s} {drawn:4d} of {total} pairs drawn (|rho| >= {cuts[a]:.3f}), "
              f"{int((rho_drawn < 0).sum())} of them negative")
    cbt = fig.colorbar(sc, cax=cax_t, orientation="horizontal")
    cbt.set_label("Temperature [$^\\circ$C]", size=PLOT.colorbar_label_size)
    cbe = fig.colorbar(lc, cax=cax_rho)
    cbe.set_label(r"$|\rho_{ij}|$", size=PLOT.colorbar_label_size)

    # Layout. `ov` is how far labels reach past the panels and bars: left/right per half, and
    # top/bottom shared, so the two cards come out equally tall.
    halves = ([ax_a, *ax_ts, cax_ts], [ax_d, ax_e, ax_f, cax_t, cax_rho])
    fixed_a = gap_ab + 2 * side + gap_sq
    fixed_b = 3 * w_map + 2 * gap + rbar_gap + rbar_w

    def place(ov):
        """Put every axes for the overhangs `ov`; return the height, cards and frames."""
        forecast_w = (width - 2 * margin - 4 * pad - split - fixed_a - fixed_b
                      - ov["left"][0] - ov["right"][0] - ov["left"][1] - ov["right"][1])
        x_a = margin + pad + ov["left"][0]
        x_b = x_a + forecast_w + gap_ab
        x_c = x_b + side + gap_sq
        end_a = x_c + side
        x_d = end_a + ov["right"][0] + 2 * pad + split + ov["left"][1]
        x_e, x_f = x_d + w_map + gap, x_d + 2 * (w_map + gap)
        x_r = x_f + w_map + rbar_gap
        end_b = x_r + rbar_w
        y_cb = margin + pad + ov["bottom"]
        y0 = y_cb + cbar_h + cbar_pad
        height = y0 + h + ov["top"] + pad + margin
        fig.set_size_inches(width, height)
        def put(ax, x, wd, y=y0, ht=h):
            ax.set_position([x / width, y / height, wd / width, ht / height])
        put(ax_a, x_a, forecast_w)
        for ax, x in zip(ax_ts, (x_b, x_c)):
            put(ax, x, side)
        for ax, x in zip((ax_d, ax_e, ax_f), (x_d, x_e, x_f)):
            put(ax, x, w_map)
        put(cax_ts, x_b, end_a - x_b, y_cb, cbar_h)
        put(cax_t, x_d, x_f + w_map - x_d, y_cb, cbar_h)
        put(cax_rho, x_r, rbar_w)
        frames = [(x_a, end_a, y_cb, y0 + h), (x_d, end_b, y_cb, y0 + h)]
        cards = [(x_a - ov["left"][0] - pad, end_a + ov["right"][0] + pad),
                 (x_d - ov["left"][1] - pad, end_b + ov["right"][1] + pad)]
        return height, forecast_w, cards, frames

    def measure(frames):
        """The overhangs of the current placement: labels past each half's frame, inches."""
        fig.canvas.draw()
        r = fig.canvas.get_renderer()
        ext = []
        for axes, (x0, x1, y0_, y1) in zip(halves, frames):
            # Decorations (ticks, labels, titles) and the frame, plus the text written outside
            # an axes -- but not the other children: a map's coastlines extend far beyond its
            # clip box.
            bb = Bbox.union([ax.get_tightbbox(r, bbox_extra_artists=extra.get(ax, []))
                             for ax in axes])
            ext.append((x0 - bb.x0 / fig.dpi, bb.x1 / fig.dpi - x1,
                        y0_ - bb.y0 / fig.dpi, bb.y1 / fig.dpi - y1))
        return {"left": (max(ext[0][0], 0.0), max(ext[1][0], 0.0)),
                "right": (max(ext[0][1], 0.0), max(ext[1][1], 0.0)),
                "bottom": max(ext[0][2], ext[1][2], 0.0), "top": max(ext[0][3], ext[1][3], 0.0)}

    guess = {"left": (0.2, 0.0), "right": (0.05, 0.37), "bottom": 0.3, "top": 0.16}
    ov = measure(place(guess)[3])
    height, forecast_w, cards, frames = place(ov)

    if boxed:
        # A canvas-sized axes in inches behind everything, so the corner radius is round.
        bg = fig.add_axes([0, 0, 1, 1], zorder=-10)
        bg.set_xlim(0, width)
        bg.set_ylim(0, height)
        bg.axis("off")
        for x0, x1 in cards:
            bg.add_patch(FancyBboxPatch(
                (x0, margin), x1 - x0, height - 2 * margin,
                boxstyle=f"round,pad=0,rounding_size={CARD_ROUND}", fc=CARD_FC, ec=CARD_EC,
                lw=PLOT.axes_line_width))

    print(f"{cs[arms[0]]['date']:%Y-%m-%d} (case {case_index}): "
          f"{int(cs[arms[0]]['mask'].sum())}/{obs.size} stations reported | "
          f"figure {width:.2f} x {height:.3f} in | panels {h:.2f} in high: forecast "
          f"{forecast_w:.3f}, correlation {side:.3f}, maps {w_map:.3f} in wide")
    print("label overhangs [in]: " + ", ".join(
        f"{k} {v[0]:.3f}/{v[1]:.3f}" if isinstance(v, tuple) else f"{k} {v:.3f}"
        for k, v in ov.items()))
    return fig


fig = plot_overview(ts)                     # boxed, every panel labelled by name
savefig(fig, "overview_etth2_euppbench")
plt.show()

#### Presentation copy — the two halves stacked

The overview above with the halves **on top of each other**: ETTh2 (forecast and the two split
correlation panels) above, EUPPBench (observed field and the two arms' $\mu$ with their
correlation edges) below, without the grey cards. Same data, colormaps, cuts and labels as
`plot_overview`; only the arrangement differs. Both rows span the full `STACKED_WIDTH`: the
forecast panel takes what the ETTh2 row leaves beside its two `STACKED_TS_H` squares, and the
maps are as large as the width allows at the domain shape `MAIN_FIG_ASPECT` gives them, with the
station markers grown along with them (their area with the map's width, so the correlation edges
stay visible between them). Written to
`figures/presentation/` (git-ignored), never over the paper figure.

In [ ]:
from matplotlib.collections import PathCollection

#: Presentation copy of the overview: canvas width and the height of the ETTh2 row, inches.
#: The EUPPBench row takes the height at which its three maps span the same width.
STACKED_WIDTH, STACKED_TS_H = 4.5, 1.2


def plot_overview_stacked(ts, case_index=EDGE_CASE, arms=None, geo=None, quantile=RHO_QUANTILE,
                          aspect_scale=MAIN_FIG_ASPECT, width=STACKED_WIDTH, ts_h=STACKED_TS_H,
                          context_h=OVERVIEW_CONTEXT_H,
                          triangle_labels=OVERVIEW_TRIANGLE_LABELS):
    '''`plot_overview` with the halves stacked -- ETTh2 on top, EUPPBench below -- and no cards.

    Placed in inches for the same reason as `plot_overview` (fixed-aspect GeoAxes), with the
    label overhangs of each row measured off a first placement.
    '''
    geo = GEO if geo is None else geo
    arms = list(EDGE_ARMS if arms is None else arms)
    extent = map_extent(geo)
    map_ratio = aspect_scale * (extent[3] - extent[2]) / (extent[1] - extent[0])   # h / w
    side = ts_h                                                          # correlation squares
    gap_ab = 0.17 if triangle_labels == "inside" else 0.20   # room for "Model"
    gap_sq = 0.17 if triangle_labels == "outside" else 0.08
    gap, rbar_gap, rbar_w = 0.08, 0.05, 0.053
    cbar_pad, cbar_h = 0.04, 0.056
    margin, row_gap = 0.02, 0.14            # canvas edge to labels; ETTh2 labels to EUPPBench titles

    fig = plt.figure(figsize=(width, 4.0), layout=PlaceHolderLayoutEngine(True, True))
    geo_kw = {"projection": PROJECTION} if HAS_CARTOPY else {}
    ax_a = fig.add_axes([0, 0, 0.1, 0.1])
    ax_ts = [fig.add_axes([0, 0, 0.1, 0.1]) for _ in arms]
    ax_maps = [fig.add_axes([0, 0, 0.1, 0.1], **geo_kw) for _ in range(3)]
    cax_ts, cax_t, cax_rho = [fig.add_axes([0, 0, 0.1, 0.1]) for _ in range(3)]

    # ETTh2: band, input, truth, and SCORE's mean dashed on top of the truth (as in `plot_overview`).
    w = ts[OURS]
    color, dt = ARM_COLORS[OURS], w["step_hours"]
    T = len(w["target"])
    lead = np.arange(1, T + 1) * dt
    n_ctx = int(round(context_h / dt))
    ax_a.fill_between(lead, w["mean"] - 2 * w["std"], w["mean"] + 2 * w["std"], color=color,
                      alpha=PLOT.band_alpha, lw=0, zorder=1)
    ax_a.plot(np.arange(-n_ctx + 1, 1) * dt, w["context"][-n_ctx:],
              color=pc.COLORS["guide"], lw=0.8, zorder=2)
    ax_a.plot(lead, w["target"], color="k", lw=0.8, zorder=3)
    ax_a.plot(lead, w["mean"], color=color, lw=1.0, ls=(0, (3, 1.5)), zorder=4)
    ax_a.axvline(0, color=pc.COLORS["guide"], lw=0.5, ls=":", zorder=0)
    ax_a.set_xlim(-n_ctx * dt, T * dt)
    hours = [-48, 0, 48, 96]
    ax_a.set_xticks(hours, labels=[f"${hr:d}$h" for hr in hours])
    overview_title(ax_a, "ETTh2", loc="center")

    imgs = [split_triangle(ts[a]["model_corr"], ts[a]["emp_corr"]) for a in arms]
    vmin = min(0.0, np.floor(10 * min(np.nanmin(im) for im in imgs)) / 10)
    norm = plt.Normalize(vmin, 1.0)
    extra = {}                              # text outside an axes, for `measure` below
    for k, (ax, a, img) in enumerate(zip(ax_ts, arms, imgs)):
        n = img.shape[0]
        im = ax.imshow(img, origin="lower", cmap=CMAP_CORR, norm=norm, aspect="auto",
                       extent=(0.5, n + 0.5, 0.5, n + 0.5), interpolation="nearest",
                       rasterized=True)
        ax.plot([0.5, n + 0.5], [0.5, n + 0.5], color=ink_on(CMAP_CORR, 1.0), lw=0.3)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.grid(False)
        overview_title(ax, SHORT_LABELS[a], loc="center")
        if triangle_labels == "inside":
            for x, y, text in [(0.24, 0.76, "Model"), (0.76, 0.24, "Empirical")]:
                r, c = int(y * n), int(x * n)
                local = img[max(r - 6, 0):r + 6, max(c - 6, 0):c + 6]
                ax.text(x, y, text, transform=ax.transAxes, rotation=45, ha="center",
                        va="center", fontsize=PLOT.annotation_size - 1,
                        color=ink_on(CMAP_CORR, norm(np.nanmedian(local))))
        else:
            sides = [(0.0, 1.0, "Model", "right", "top", -2, 90),
                     (1.0, 0.0, "Empirical", "left", "bottom", 2, 270)]
            if triangle_labels == "once":
                sides = [sd for sd, keep in zip(sides, (k == 0, k == len(arms) - 1)) if keep]
            for x, y, text, ha, va, dx, rot in sides:
                extra.setdefault(ax, []).append(ax.annotate(
                    text, (x, y), xycoords="axes fraction", xytext=(dx, 0),
                    textcoords="offset points", rotation=rot, ha=ha, va=va,
                    fontsize=PLOT.annotation_size - 1, color="0.25"))
    cb = fig.colorbar(im, cax=cax_ts, orientation="horizontal",
                      format=mticker.FuncFormatter(lambda v, _: f"${v:g}$"))   # math minus
    cb.set_ticks(np.round(np.arange(vmin, 1.0 + 1e-9, 0.2), 1))
    cb.set_label(r"Correlation $\rho$", size=PLOT.colorbar_label_size)

    # EUPPBench: the observed field, then each arm's mu with its strongest correlation edges.
    cs = get_cases(case_index, arms)
    obs = cs[arms[0]]["obs"]
    pool = np.concatenate([obs[np.isfinite(obs)]] + [cs[a]["mu"] for a in arms])
    vmin_t, vmax_t = float(pool.min()), float(pool.max())
    cuts = {a: (edge_cut(cs[a]["corr"], quantile) if RHO_THRESHOLD is None else RHO_THRESHOLD)
            for a in arms}
    rho_norm = (min(cuts.values()),
                max(float(np.abs(off_diagonal(cs[a]["corr"])).max()) for a in arms))
    sc = station_map(ax_maps[0], geo, obs, cmap=CMAP_T, vmin=vmin_t, vmax=vmax_t, size=8)
    stretch_domain(ax_maps[0], aspect_scale)
    overview_title(ax_maps[0], "EUPPBench", loc="center")
    for ax, a in zip(ax_maps[1:], arms):
        station_map(ax, geo, cs[a]["mu"], cmap=CMAP_T, vmin=vmin_t, vmax=vmax_t, size=8)
        stretch_domain(ax, aspect_scale)
        lc, drawn, total, rho_drawn = draw_correlation_edges(
            ax, geo, cs[a]["corr"], threshold=cuts[a], norm=rho_norm)
        overview_title(ax, SHORT_LABELS[a], loc="center")
        print(f"{LABELS[a]:<14s} {drawn:4d} of {total} pairs drawn (|rho| >= {cuts[a]:.3f}), "
              f"{int((rho_drawn < 0).sum())} of them negative")
    cbt = fig.colorbar(sc, cax=cax_t, orientation="horizontal")
    cbt.set_label("Temperature [$^\\circ$C]", size=PLOT.colorbar_label_size)
    cbe = fig.colorbar(lc, cax=cax_rho)
    cbe.set_label(r"$|\rho_{ij}|$", size=PLOT.colorbar_label_size)

    rows = ([ax_a, *ax_ts, cax_ts], [*ax_maps, cax_t, cax_rho])

    def place(ov):
        '''Put every axes for the overhangs `ov` = per row (left, right, bottom, top).'''
        (lt, rt, bt, tt), (lb, rb, bb, tb) = ov
        # EUPPBench row at the bottom: three maps and the |rho| bar fill the width.
        x_d = margin + lb
        w_map = (width - margin - rb - x_d - 2 * gap - rbar_gap - rbar_w) / 3
        h_map = w_map * map_ratio
        y_cb_b = margin + bb
        y_b = y_cb_b + cbar_h + cbar_pad
        # ETTh2 row above it: the squares right-aligned, the forecast takes the rest.
        y_cb_t = y_b + h_map + tb + row_gap + bt
        y_t = y_cb_t + cbar_h + cbar_pad
        height = y_t + side + tt + margin
        x_a = margin + lt
        x_c = width - margin - rt - side
        x_b = x_c - gap_sq - side
        forecast_w = x_b - gap_ab - x_a
        fig.set_size_inches(width, height)
        def put(ax, x, wd, y, ht):
            ax.set_position([x / width, y / height, wd / width, ht / height])
        put(ax_a, x_a, forecast_w, y_t, side)
        for ax, x in zip(ax_ts, (x_b, x_c)):
            put(ax, x, side, y_t, side)
        put(cax_ts, x_b, x_c + side - x_b, y_cb_t, cbar_h)
        xs = [x_d + k * (w_map + gap) for k in range(3)]
        for ax, x in zip(ax_maps, xs):
            put(ax, x, w_map, y_b, h_map)
        put(cax_t, x_d, xs[-1] + w_map - x_d, y_cb_b, cbar_h)
        x_r = xs[-1] + w_map + rbar_gap
        put(cax_rho, x_r, rbar_w, y_b, h_map)
        frames = [(x_a, x_c + side, y_cb_t, y_t + side), (x_d, x_r + rbar_w, y_cb_b, y_b + h_map)]
        return height, forecast_w, w_map, frames

    def measure(frames):
        '''How far each row's labels reach past its panels and bars, inches (as in `plot_overview`).'''
        fig.canvas.draw()
        r = fig.canvas.get_renderer()
        ov = []
        for axes, (x0, x1, y0, y1) in zip(rows, frames):
            bb = Bbox.union([ax.get_tightbbox(r, bbox_extra_artists=extra.get(ax, []))
                             for ax in axes])
            ov.append(tuple(max(v, 0.0) for v in (x0 - bb.x0 / fig.dpi, bb.x1 / fig.dpi - x1,
                                                   y0 - bb.y0 / fig.dpi, bb.y1 / fig.dpi - y1)))
        return ov

    ov = [(0.2, 0.11, 0.32, 0.12), (0.0, 0.37, 0.32, 0.12)]
    for _ in range(2):                      # the map size moves with the overhangs, so twice
        ov = measure(place(ov)[3])
    height, forecast_w, w_map, _ = place(ov)

    # Station markers were drawn at the paper's size for the paper's map width. Their area grows
    # with the map's width rather than its area: at full area scaling they would cover the map
    # as in the paper, but the edges and coastlines keep their line widths, so the markers
    # would bury the correlation edges the two method panels are about.
    grow = w_map / (OVERVIEW_PANEL_H / map_ratio)
    for ax in ax_maps:
        for coll in ax.collections:
            if isinstance(coll, PathCollection):
                coll.set_sizes(coll.get_sizes() * grow)

    print(f"{cs[arms[0]]['date']:%Y-%m-%d} (case {case_index}): figure {width:.2f} x "
          f"{height:.3f} in | ETTh2 row {side:.2f} in high (forecast {forecast_w:.3f} in wide), "
          f"maps {w_map:.3f} x {w_map * map_ratio:.3f} in, markers x{grow:.2f}")
    return fig


fig = plot_overview_stacked(ts)
print(f"wrote {pc.save(fig, pc.PRESENTATION_DIR / 'overview_etth2_euppbench.pdf')}")
plt.show()
